In [23]:
# ============================================================
# OUTLIER DETECTION + CLEANED SPEED DATA + NEW V85 CALCULATION
# ============================================================

# Step 1: Import libraries

import pandas as pd
import numpy as np
import os
from pathlib import Path

In [24]:
# Step 2: Set folder path

folder_path = r"D:\MSC THESIS AAVAS\AFTER MID DEFENSE\Whole data for cleaning"

# Outputs will be saved in the same folder
output_folder = folder_path

print("Input folder:")
print(folder_path)
print("\nOutput folder:")
print(output_folder)

Input folder:
D:\MSC THESIS AAVAS\AFTER MID DEFENSE\Whole data for cleaning

Output folder:
D:\MSC THESIS AAVAS\AFTER MID DEFENSE\Whole data for cleaning


In [25]:
# Step 3: Function for outlier detection using IQR method

def clean_speed_data(df, speed_col="Speeds"):
    """
    This function:
    1. Reads speed values
    2. Detects outliers using IQR method
    3. Removes outliers
    4. Calculates new V85 speed after cleaning
    """

    # Clean column names
    df.columns = df.columns.astype(str).str.strip()

    # Convert speed values into numeric format
    df[speed_col] = pd.to_numeric(df[speed_col], errors="coerce")

    # Remove blank or non-numeric speed values
    df = df.dropna(subset=[speed_col]).copy()

    # Calculate Q1, Q3 and IQR
    Q1 = df[speed_col].quantile(0.25)
    Q3 = df[speed_col].quantile(0.75)
    IQR = Q3 - Q1

    # Calculate lower and upper limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # Mark each value as Accepted or Outlier
    df["Outlier_Status"] = np.where(
        (df[speed_col] < lower_limit) | (df[speed_col] > upper_limit),
        "Outlier",
        "Accepted"
    )

    # Keep only accepted values
    cleaned_df = df[df["Outlier_Status"] == "Accepted"].copy()

    # Calculate original V85 before outlier removal
    original_v85 = np.percentile(df[speed_col], 85)

    # Calculate new V85 after outlier removal
    if len(cleaned_df) > 0:
        new_v85 = np.percentile(cleaned_df[speed_col], 85)
    else:
        new_v85 = np.nan

    # Summary values
    summary = {
        "Original_Count": len(df),
        "Cleaned_Count": len(cleaned_df),
        "Outliers_Removed": len(df) - len(cleaned_df),
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Limit": lower_limit,
        "Upper_Limit": upper_limit,
        "Original_V85": original_v85,
        "New_V85": new_v85
    }

    return df, cleaned_df, summary

In [26]:
# Step 4: Read all Excel files and process all sheets

summary_list = []

# Read all Excel files in the folder
excel_files = list(Path(folder_path).glob("*.xlsx"))

# To avoid reading previously generated output files again
excel_files = [
    file for file in excel_files
    if not file.name.startswith("Outlier_")
    and not file.name.startswith("Cleaned_")
    and not file.name.startswith("Final_")
]

print(f"Total Excel files found: {len(excel_files)}")

for file_path in excel_files:

    file_name = file_path.name
    print(f"\nProcessing file: {file_name}")

    # Read all sheets from current Excel file
    all_sheets = pd.read_excel(file_path, sheet_name=None)

    checked_sheets = {}
    cleaned_sheets = {}

    for sheet_name, df in all_sheets.items():

        print(f"  Processing sheet: {sheet_name}")

        # Clean column names
        df.columns = df.columns.astype(str).str.strip()

        # Check if required speed column exists
        if "Speeds" not in df.columns:
            print(f"    Skipped: 'Speeds' column not found")
            continue

        # Apply outlier detection
        checked_df, cleaned_df, summary = clean_speed_data(df, speed_col="Speeds")

        # Store sheets
        checked_sheets[sheet_name] = checked_df
        cleaned_sheets[sheet_name] = cleaned_df

        # Add result to summary table
        summary_list.append({
            "File_Name": file_name,
            "Sheet_Name": sheet_name,
            "Original_Count": summary["Original_Count"],
            "Cleaned_Count": summary["Cleaned_Count"],
            "Outliers_Removed": summary["Outliers_Removed"],
            "Q1": round(summary["Q1"], 2),
            "Q3": round(summary["Q3"], 2),
            "IQR": round(summary["IQR"], 2),
            "Lower_Limit": round(summary["Lower_Limit"], 2),
            "Upper_Limit": round(summary["Upper_Limit"], 2),
            "Original_V85_km_hr": round(summary["Original_V85"], 2),
            "New_V85_km_hr": round(summary["New_V85"], 2)
        })

    # Save outlier checked file in same folder
    checked_output_path = os.path.join(
        output_folder,
        "Outlier_Checked_" + file_name
    )

    if checked_sheets:
        with pd.ExcelWriter(checked_output_path, engine="openpyxl") as writer:
            for sheet_name, data in checked_sheets.items():
                data.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    # Save cleaned file in same folder
    cleaned_output_path = os.path.join(
        output_folder,
        "Cleaned_" + file_name
    )

    if cleaned_sheets:
        with pd.ExcelWriter(cleaned_output_path, engine="openpyxl") as writer:
            for sheet_name, data in cleaned_sheets.items():
                data.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print("\nAll files processed successfully.")

Total Excel files found: 5

Processing file: Whole_data_bp.xlsx
  Processing sheet: Point 1
  Processing sheet: Point 2
  Processing sheet: Point 3
  Processing sheet: Point 4
  Processing sheet: Point 5
  Processing sheet: Point 6
  Processing sheet: Point 7
  Processing sheet: Point 8
  Processing sheet: Point 9
  Processing sheet: Point 10
  Processing sheet: Point 11
  Processing sheet: Point 12
  Processing sheet: Point 13
  Processing sheet: Point 14
  Processing sheet: Point 15
  Processing sheet: Point 16
  Processing sheet: Point 17
  Processing sheet: Point 18
  Processing sheet: Point 19
  Processing sheet: Point 20
  Processing sheet: Point 21
  Processing sheet: Point 22
  Processing sheet: Point 23
  Processing sheet: Point 24
  Processing sheet: Point 25
  Processing sheet: Point 26
  Processing sheet: Point 27
  Processing sheet: Point 28
  Processing sheet: Point 29
  Processing sheet: Point 30
  Processing sheet: Point 31
  Processing sheet: Point 32
  Processing shee

Exception ignored in: <function ZipFile.__del__ at 0x000002DA3BB78400>
Traceback (most recent call last):
  File "C:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\zipfile\__init__.py", line 2001, in __del__
    self.close()
  File "C:\Users\ACER\AppData\Local\Programs\Python\Python313\Lib\zipfile\__init__.py", line 2018, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file



Processing file: Whole_data_NN.xlsx
  Processing sheet: P1
  Processing sheet: P2
  Processing sheet: P3
  Processing sheet: P4
  Processing sheet: P5
  Processing sheet: P6
  Processing sheet: P7
  Processing sheet: P8
  Processing sheet: P9
  Processing sheet: P10
  Processing sheet: P11
  Processing sheet: P12
  Processing sheet: P13
  Processing sheet: P14
  Processing sheet: P15
  Processing sheet: P16
  Processing sheet: P18
  Processing sheet: P19
  Processing sheet: P20
  Processing sheet: P21
  Processing sheet: P22
  Processing sheet: P23
  Processing sheet: P24
  Processing sheet: P25
  Processing sheet: P26
  Processing sheet: P27
  Processing sheet: P28
  Processing sheet: P29
  Processing sheet: P30
  Processing sheet: P31
  Processing sheet: P32
  Processing sheet: P33
  Processing sheet: P34
  Processing sheet: P35
  Processing sheet: P36

Processing file: Whole_data_NNM.xlsx
  Processing sheet: Point 1
  Processing sheet: Point 2
  Processing sheet: Point 3
  Processi

In [27]:
# Step 5: Create and save final V85 summary table

summary_df = pd.DataFrame(summary_list)

summary_output_path = os.path.join(
    output_folder,
    "Final_V85_Summary_After_Outlier_Removal.xlsx"
)

summary_df.to_excel(summary_output_path, index=False)

print("Final summary saved at:")
print(summary_output_path)

summary_df

Final summary saved at:
D:\MSC THESIS AAVAS\AFTER MID DEFENSE\Whole data for cleaning\Final_V85_Summary_After_Outlier_Removal.xlsx


,File_Name,Sheet_Name,Original_Count,Cleaned_Count,Outliers_Removed,Q1,Q3,IQR,Lower_Limit,Upper_Limit,Original_V85_km_hr,New_V85_km_hr
0,Whole_data_bp.xlsx,Point 1,105,104,1,39.00,53.00,14.00,18.00,74.00,57.00,57.00
1,Whole_data_bp.xlsx,Point 2,106,102,4,27.00,32.00,5.00,19.50,39.50,33.25,33.00
2,Whole_data_bp.xlsx,Point 3,103,102,1,32.00,41.00,9.00,18.50,54.50,43.00,43.00
3,Whole_data_bp.xlsx,Point 4,102,102,0,34.00,45.00,11.00,17.50,61.50,48.00,48.00
4,Whole_data_bp.xlsx,Point 5,106,104,2,28.00,35.00,7.00,17.50,45.50,36.00,36.00
...,...,...,...,...,...,...,...,...,...,...,...,...
172,Whole_data_yam_bag_o.xlsx,Point25,140,136,4,40.00,54.25,14.25,18.62,75.62,58.15,58.75
173,Whole_data_yam_bag_o.xlsx,Point 26,110,110,0,37.25,51.75,14.50,15.50,73.50,54.00,54.00
174,Whole_data_yam_bag_o.xlsx,Point27,140,134,6,43.00,55.00,12.00,25.00,73.00,59.15,60.00
175,Whole_data_yam_bag_o.xlsx,Point 28,107,107,0,27.50,40.00,12.50,8.75,58.75,44.00,44.00
